# Tutorial 11E: Statistical Testing and Model Comparison

**Module**: Stochastic Frontier Analysis | **Difficulty**: Intermediate-Advanced | **Duration**: 3-4 hours

**Prerequisites**: Tutorials 01-04; Hypothesis testing; Bootstrap methods

## Learning Objectives

By the end of this tutorial, you will be able to:

1. **Test** for presence of inefficiency (SFA vs OLS)
2. **Compare** distributional assumptions (nested and non-nested)
3. **Select** optimal panel model specifications
4. **Test** production hypotheses (RTS, functional form)
5. **Apply** LR, Wald, and Hausman tests
6. **Implement** bootstrap inference for SFA
7. **Conduct** comprehensive model selection workflows
8. **Validate** model assumptions and diagnostics

## Table of Contents

1. [Setup and Data Loading](#1-setup-and-data-loading)
2. [Test for Inefficiency Presence](#2-test-for-inefficiency-presence)
3. [Distribution Comparison](#3-distribution-comparison)
4. [Panel Model Selection](#4-panel-model-selection)
5. [Returns to Scale Testing](#5-returns-to-scale-testing)
6. [Functional Form Testing](#6-functional-form-testing)
7. [Bootstrap Inference](#7-bootstrap-inference)
8. [Model Selection Workflow](#8-model-selection-workflow)
9. [Export Results](#9-export-results)
10. [Exercises](#10-exercises)
11. [Summary](#11-summary)
12. [References](#12-references)

## Datasets

| Dataset | Type | N | Variables | Purpose |
|---------|------|---|-----------|--------|
| `dairy_farm.csv` | Cross-section | 500 farms | log_milk, log_cows, log_feed, log_land, log_labor, organic, breed, cooperative | Distribution testing, RTS, functional form |
| `telecom_panel.csv` | Panel | 40 firms x 15 years | log_output, log_labor, log_capital, log_spectrum, technology, market_share | Panel model selection |

## 1. Setup and Data Loading

<a id="1-setup-and-data-loading"></a>

In [ ]:
# ============================================================
# Section 1: Setup and Data Loading
# ============================================================

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
from pathlib import Path

import statsmodels.api as sm
from scipy import stats

from panelbox.frontier import (
    StochasticFrontier,
    add_translog,
    compare_nested_distributions,
    inefficiency_presence_test,
    skewness_test,
)
from panelbox.frontier.bootstrap import SFABootstrap
from panelbox.frontier.panel_utils import (
    lr_test_bc92_eta_constant,
    lr_test_kumbhakar_constant,
)

np.random.seed(42)

plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["legend.fontsize"] = 10
pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 4)

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
FIGURES_DIR = BASE_DIR / "outputs" / "figures" / "05_testing"
TABLES_DIR_LATEX = BASE_DIR / "outputs" / "tables" / "latex"
TABLES_DIR_HTML = BASE_DIR / "outputs" / "tables" / "html"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR_LATEX.mkdir(parents=True, exist_ok=True)
TABLES_DIR_HTML.mkdir(parents=True, exist_ok=True)

print("Setup complete!")

In [ ]:
# Load the dairy farm cross-section dataset
data = pd.read_csv(DATA_DIR / "dairy_farm.csv")

print(f"Dataset shape: {data.shape}")
print(f"Variables: {data.columns.tolist()}")
print(f"\nNumber of farms: {data['farm_id'].nunique()}")
print("\nFirst 5 observations:")
display(data.head())

print("\nSummary Statistics:")
display(data.describe().round(4))

### Understanding the Variables

| Variable | Description | Role |
|----------|-------------|------|
| `farm_id` | Farm identifier | ID |
| `log_milk` | Log of milk output | **Dependent variable** |
| `log_cows` | Log of number of cows | **Input** |
| `log_feed` | Log of feed expenditure | **Input** |
| `log_land` | Log of land area | **Input** |
| `log_labor` | Log of labor hours | **Input** |
| `organic` | Organic farm (1=yes) | Control |
| `breed` | Breed type (holstein/jersey/mixed) | Control |
| `cooperative` | Cooperative member (1=yes) | Control |

---

## 2. Test for Inefficiency Presence (20 min)

<a id="2-test-for-inefficiency-presence"></a>

### Why Test for Inefficiency?

Before estimating an SFA model, we should first ask: **is there evidence of inefficiency at all?** If not, OLS is adequate and the SFA model is over-specified.

### The Hypothesis

$$H_0: \sigma^2_u = 0 \quad \text{(OLS adequate)}$$
$$H_1: \sigma^2_u > 0 \quad \text{(Inefficiency present)}$$

### The Mixed Chi-Square Problem

Because $\sigma^2_u \geq 0$ is a **boundary constraint**, the standard LR test distribution does not apply. Under $H_0$, the LR statistic follows a **mixed chi-square** distribution (Kodde & Palm, 1986):

- For half-normal or exponential: $LR \sim \frac{1}{2}\chi^2(0) + \frac{1}{2}\chi^2(1)$
- Critical value at 5%: **2.706** (not 3.841 as for standard $\chi^2(1)$)

### Skewness Diagnostic

For a **production frontier**, OLS residuals should have **negative skewness** (because $u \geq 0$ pulls residuals left). Positive skewness may indicate:
- Wrong frontier type (should be cost instead of production)
- No inefficiency present
- Distributional misspecification

In [ ]:
# Step 1: Estimate OLS baseline
exog_vars = ["log_cows", "log_feed", "log_land", "log_labor"]
X_ols = sm.add_constant(data[exog_vars])
y_ols = data["log_milk"]
ols_model = sm.OLS(y_ols, X_ols).fit()

print("=" * 60)
print("OLS BASELINE MODEL")
print("=" * 60)
print(f"Log-likelihood: {ols_model.llf:.4f}")
print(f"R-squared:      {ols_model.rsquared:.4f}")
print("\nCoefficients:")
for name, coef, se in zip(ols_model.params.index, ols_model.params.values, ols_model.bse.values):
    print(f"  {name:<12}: {coef:>8.4f}  (SE: {se:.4f})")

In [ ]:
# Step 2: Estimate SFA model (half-normal)
sfa_model = StochasticFrontier(
    data=data,
    depvar="log_milk",
    exog=exog_vars,
    frontier="production",
    dist="half_normal",
)
sfa_result = sfa_model.fit()

print("=" * 60)
print("SFA MODEL (Half-Normal)")
print("=" * 60)
print(f"Log-likelihood: {sfa_result.loglik:.4f}")
print(f"sigma_v:        {sfa_result.sigma_v:.4f}")
print(f"sigma_u:        {sfa_result.sigma_u:.4f}")
print(f"lambda:         {sfa_result.lambda_param:.4f}")
print(f"gamma:          {sfa_result.gamma:.4f}")
print(f"Mean TE:        {sfa_result.mean_efficiency:.4f}")

In [ ]:
# Step 3: Test for inefficiency presence
test_result = inefficiency_presence_test(
    loglik_sfa=sfa_result.loglik,
    loglik_ols=ols_model.llf,
    residuals_ols=ols_model.resid.values,
    frontier_type="production",
    distribution="half_normal",
)

print("=" * 60)
print("TEST FOR PRESENCE OF INEFFICIENCY")
print("=" * 60)
print("H0: sigma_u^2 = 0 (OLS adequate)")
print("H1: sigma_u^2 > 0 (Inefficiency present)")
print(f"\nLog-likelihood (OLS): {ols_model.llf:.4f}")
print(f"Log-likelihood (SFA): {sfa_result.loglik:.4f}")
print(f"\nLR Statistic:     {test_result['lr_statistic']:.4f}")
print(f"Distribution:     Mixed chi-square ({test_result['test_type']})")
print(f"P-value:          {test_result['pvalue']:.4f}")
print("\nCritical Values:")
for level, cv in test_result["critical_values"].items():
    reject = "REJECT" if test_result["lr_statistic"] > cv else "fail to reject"
    print(f"  {level}: {cv:.3f}  -> {reject}")
print(f"\nConclusion: {test_result['conclusion']}")
print(f"{test_result['interpretation']}")

In [ ]:
# Step 4: Skewness diagnostic
skew_result = skewness_test(
    residuals=ols_model.resid.values,
    frontier_type="production",
)

print("=" * 60)
print("SKEWNESS DIAGNOSTIC")
print("=" * 60)
print(f"Skewness of OLS residuals: {skew_result['skewness']:.4f}")
print(f"Expected sign for production frontier: {skew_result['expected_sign']}")
print(f"Correct sign: {skew_result['correct_sign']}")
if skew_result.get("warning"):
    print(f"Warning: {skew_result['warning']}")
else:
    print("No warning: Skewness is consistent with production frontier.")

# Also run scipy skewness test
scipy_skew_stat, scipy_skew_pval = stats.skewtest(ols_model.resid.values)
print("\nScipy skewtest:")
print(f"  Statistic: {scipy_skew_stat:.4f}")
print(f"  P-value:   {scipy_skew_pval:.4f}")

In [ ]:
# Visualize the inefficiency test
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: OLS residual distribution with skewness
residuals = ols_model.resid.values
axes[0].hist(
    residuals, bins=40, density=True, alpha=0.7, color="steelblue", edgecolor="black", linewidth=0.5
)
kde = stats.gaussian_kde(residuals)
x_range = np.linspace(residuals.min(), residuals.max(), 200)
axes[0].plot(x_range, kde(x_range), "r-", linewidth=2, label="KDE")

# Normal overlay
x_norm = np.linspace(residuals.min(), residuals.max(), 200)
axes[0].plot(
    x_norm,
    stats.norm.pdf(x_norm, residuals.mean(), residuals.std()),
    "g--",
    linewidth=2,
    label="Normal",
)
axes[0].axvline(0, color="black", linewidth=1, linestyle="-")
axes[0].set_xlabel("OLS Residuals", fontsize=12)
axes[0].set_ylabel("Density", fontsize=12)
axes[0].set_title(
    f"OLS Residuals (Skewness = {skew_result['skewness']:.3f})", fontsize=13, fontweight="bold"
)
axes[0].legend(fontsize=10)

# Panel 2: LR test visualization
x_chi2 = np.linspace(0, 15, 500)
# Mixed chi-square: 0.5*chi2(1)
y_mixed = 0.5 * stats.chi2.pdf(x_chi2, df=1)
axes[1].plot(x_chi2, y_mixed, "b-", linewidth=2, label="Mixed $\\chi^2$ (half-normal)")
axes[1].fill_between(
    x_chi2, y_mixed, where=x_chi2 > 2.706, alpha=0.3, color="red", label="Rejection region (5%)"
)
axes[1].axvline(
    test_result["lr_statistic"],
    color="red",
    linewidth=2,
    linestyle="--",
    label=f"LR = {test_result['lr_statistic']:.2f}",
)
axes[1].axvline(
    2.706, color="orange", linewidth=1.5, linestyle=":", label="Critical value (5%) = 2.706"
)
axes[1].set_xlabel("LR Statistic", fontsize=12)
axes[1].set_ylabel("Density", fontsize=12)
axes[1].set_title("LR Test for Inefficiency Presence", fontsize=13, fontweight="bold")
axes[1].legend(fontsize=9)
axes[1].set_xlim(0, 15)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "inefficiency_test.png", dpi=300, bbox_inches="tight")
plt.show()

print("Figure saved: inefficiency_test.png")

*Figure: Left panel shows the distribution of OLS residuals with KDE and normal overlay. Negative skewness supports the presence of inefficiency in a production frontier. Right panel shows the LR test statistic against the mixed chi-square critical region.*

---

## 3. Distribution Comparison (25 min)

<a id="3-distribution-comparison"></a>

### Nested vs Non-Nested Comparisons

| Comparison | Type | Test |
|-----------|------|------|
| Half-normal vs Truncated normal | Nested ($\mu = 0$) | LR test |
| Half-normal vs Exponential | Non-nested | AIC/BIC, Vuong |
| Exponential vs Truncated normal | Non-nested | AIC/BIC, Vuong |

### Information Criteria

$$\text{AIC} = -2 \ln L + 2k$$
$$\text{BIC} = -2 \ln L + k \ln(N)$$

where $k$ = number of parameters, $N$ = sample size. Lower is better.

In [ ]:
# Estimate SFA with different distributions
distributions = ["half_normal", "exponential", "truncated_normal"]
models = {}

for dist in distributions:
    print(f"Estimating {dist}...")
    try:
        model = StochasticFrontier(
            data=data,
            depvar="log_milk",
            exog=exog_vars,
            frontier="production",
            dist=dist,
        )
        models[dist] = model.fit()
        print(f"  Log-L: {models[dist].loglik:.4f}, Mean TE: {models[dist].mean_efficiency:.4f}")
    except Exception as e:
        print(f"  Failed: {e}")

print(f"\nSuccessfully estimated: {list(models.keys())}")

In [ ]:
# Use the built-in compare_distributions method
dist_comparison = sfa_result.compare_distributions(
    other_results=[models[d] for d in models if d != "half_normal"]
)

print("=" * 80)
print("DISTRIBUTION COMPARISON")
print("=" * 80)
display(
    dist_comparison[
        [
            "Distribution",
            "Log-Likelihood",
            "AIC",
            "BIC",
            "σ_v",
            "σ_u",
            "λ",
            "Mean Efficiency",
            "Rank AIC",
            "Rank BIC",
        ]
    ]
)

# Best models
best_aic = dist_comparison.loc[dist_comparison["AIC"].idxmin(), "Distribution"]
best_bic = dist_comparison.loc[dist_comparison["BIC"].idxmin(), "Distribution"]
print(f"\nBest Model by AIC: {best_aic}")
print(f"Best Model by BIC: {best_bic}")

In [ ]:
# LR test: Half-normal vs Truncated normal (nested)
if "half_normal" in models and "truncated_normal" in models:
    lr_hn_tn = compare_nested_distributions(
        loglik_restricted=models["half_normal"].loglik,
        loglik_unrestricted=models["truncated_normal"].loglik,
        dist_restricted="half_normal",
        dist_unrestricted="truncated_normal",
    )

    print("=" * 60)
    print("NESTED TEST: Half-Normal vs Truncated Normal")
    print("=" * 60)
    print("H0: mu = 0 (half-normal is adequate)")
    print("H1: mu != 0 (truncated normal is needed)")
    print(f"\nLR Statistic: {lr_hn_tn['lr_statistic']:.4f}")
    print(f"P-value:      {lr_hn_tn['pvalue']:.4f}")
    print(f"Conclusion:   {lr_hn_tn['conclusion']}")
    print(f"{lr_hn_tn['interpretation']}")

In [ ]:
# Visualize distribution comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: AIC/BIC bar chart
dist_names = dist_comparison["Distribution"].values
aic_vals = dist_comparison["AIC"].values
bic_vals = dist_comparison["BIC"].values

x = np.arange(len(dist_names))
width = 0.35

bars1 = axes[0].bar(x - width / 2, aic_vals, width, label="AIC", color="steelblue", alpha=0.8)
bars2 = axes[0].bar(x + width / 2, bic_vals, width, label="BIC", color="#FF9800", alpha=0.8)

axes[0].set_xticks(x)
axes[0].set_xticklabels(dist_names, fontsize=10)
axes[0].set_ylabel("Information Criterion", fontsize=12)
axes[0].set_title("AIC and BIC by Distribution", fontsize=13, fontweight="bold")
axes[0].legend(fontsize=10)

# Highlight the best
min_aic_idx = np.argmin(aic_vals)
min_bic_idx = np.argmin(bic_vals)
bars1[min_aic_idx].set_edgecolor("red")
bars1[min_aic_idx].set_linewidth(2)
bars2[min_bic_idx].set_edgecolor("red")
bars2[min_bic_idx].set_linewidth(2)

# Panel 2: Efficiency distributions by model
for dist_name, result in models.items():
    eff = result.efficiency(estimator="bc")["efficiency"].values
    axes[1].hist(eff, bins=30, density=True, alpha=0.4, label=dist_name)

axes[1].set_xlabel("Technical Efficiency", fontsize=12)
axes[1].set_ylabel("Density", fontsize=12)
axes[1].set_title("Efficiency Distribution by Model", fontsize=13, fontweight="bold")
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "distribution_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

print("Figures saved: distribution_comparison.png")

*Figure: Left panel shows AIC and BIC values for each distributional specification (lower is better; red border marks the minimum). Right panel overlays the efficiency distributions from each model to visualize how distributional choice affects efficiency estimates.*

In [ ]:
# Efficiency correlation across distributions
eff_dict = {}
for dist_name, result in models.items():
    eff_dict[dist_name] = result.efficiency(estimator="bc")["efficiency"].values

eff_df = pd.DataFrame(eff_dict)

print("=" * 60)
print("RANK CORRELATION OF EFFICIENCY ESTIMATES")
print("=" * 60)
print("\nSpearman rank correlations:")
for i, d1 in enumerate(eff_df.columns):
    for d2 in eff_df.columns[i + 1 :]:
        rho, pval = stats.spearmanr(eff_df[d1], eff_df[d2])
        print(f"  {d1} vs {d2}: rho = {rho:.4f} (p = {pval:.4f})")

print("\nPearson correlations:")
display(eff_df.corr().round(4))

---

## 4. Panel Model Selection (30 min)

<a id="4-panel-model-selection"></a>

### Panel SFA Models

| Model | Key Feature | Time-Varying? | Test |
|-------|-------------|---------------|------|
| Pitt-Lee | Time-invariant $u_i$ | No | Baseline |
| BC92 | $u_{it} = u_i \cdot \exp(-\eta(t-T))$ | Yes | LR: $\eta = 0$ |
| Kumbhakar 1990 | Flexible logistic time pattern | Yes | LR: $b = c = 0$ |

### Testing Time-Varying Inefficiency

- **Pitt-Lee vs BC92**: $H_0: \eta = 0$ (time-invariant), $q = 1$
- **Pitt-Lee vs Kumbhakar**: $H_0: b = c = 0$ (time-invariant), $q = 2$

In [ ]:
# Load panel data
panel_data = pd.read_csv(DATA_DIR / "telecom_panel.csv")

print(f"Panel dataset shape: {panel_data.shape}")
print(f"Variables: {panel_data.columns.tolist()}")
print(f"\nNumber of firms: {panel_data['firm_id'].nunique()}")
print(f"Number of years: {panel_data['year'].nunique()}")
print(f"Year range: {panel_data['year'].min()} - {panel_data['year'].max()}")
print("\nFirst 5 observations:")
display(panel_data.head())

In [ ]:
# Estimate different panel models
panel_exog = ["log_labor", "log_capital", "log_spectrum"]
panel_models = {}

model_specs = {
    "Pitt-Lee": "pitt_lee",
    "BC92": "bc92",
    "Kumbhakar": "kumbhakar_1990",
}

for name, mtype in model_specs.items():
    print(f"Estimating {name}...")
    try:
        model = StochasticFrontier(
            data=panel_data,
            depvar="log_output",
            exog=panel_exog,
            entity="firm_id",
            time="year",
            frontier="production",
            dist="half_normal",
            model_type=mtype,
        )
        panel_models[name] = model.fit()
        print(
            f"  Log-L: {panel_models[name].loglik:.4f}, Converged: {panel_models[name].converged}"
        )
    except Exception as e:
        print(f"  Failed: {e}")

print(f"\nSuccessfully estimated: {list(panel_models.keys())}")

In [ ]:
# Panel model comparison table
panel_comparison = pd.DataFrame(
    {
        "Model": list(panel_models.keys()),
        "Log-Likelihood": [r.loglik for r in panel_models.values()],
        "AIC": [r.aic for r in panel_models.values()],
        "BIC": [r.bic for r in panel_models.values()],
        "N Parameters": [r.nparams for r in panel_models.values()],
        "sigma_v": [r.sigma_v for r in panel_models.values()],
        "sigma_u": [r.sigma_u for r in panel_models.values()],
        "Mean TE": [r.mean_efficiency for r in panel_models.values()],
    }
)

print("=" * 80)
print("PANEL MODEL COMPARISON")
print("=" * 80)
display(panel_comparison.round(4))

# Best model
best_panel_aic = panel_comparison.loc[panel_comparison["AIC"].idxmin(), "Model"]
best_panel_bic = panel_comparison.loc[panel_comparison["BIC"].idxmin(), "Model"]
print(f"\nBest by AIC: {best_panel_aic}")
print(f"Best by BIC: {best_panel_bic}")

In [ ]:
# LR tests for time-varying inefficiency
print("=" * 60)
print("LR TESTS FOR TIME-VARYING INEFFICIENCY")
print("=" * 60)

# Test 1: Pitt-Lee vs BC92 (H0: eta = 0)
if "Pitt-Lee" in panel_models and "BC92" in panel_models:
    lr_pl_bc92 = lr_test_bc92_eta_constant(
        loglik_bc92=panel_models["BC92"].loglik,
        loglik_pitt_lee=panel_models["Pitt-Lee"].loglik,
    )
    print("\nTest 1: Pitt-Lee vs BC92")
    print("  H0: eta = 0 (time-invariant inefficiency)")
    print(f"  LR Statistic: {lr_pl_bc92['LR_stat']:.4f}")
    print(f"  P-value:      {lr_pl_bc92['p_value']:.4f}")
    print(f"  Reject H0:    {lr_pl_bc92['reject_H0']}")
    if lr_pl_bc92["reject_H0"]:
        print("  -> Time-varying inefficiency is significant")
    else:
        print("  -> Time-invariant model is adequate")

# Test 2: Pitt-Lee vs Kumbhakar (H0: b = c = 0)
if "Pitt-Lee" in panel_models and "Kumbhakar" in panel_models:
    lr_pl_kumb = lr_test_kumbhakar_constant(
        loglik_kumbhakar=panel_models["Kumbhakar"].loglik,
        loglik_pitt_lee=panel_models["Pitt-Lee"].loglik,
    )
    print("\nTest 2: Pitt-Lee vs Kumbhakar 1990")
    print("  H0: b = c = 0 (time-invariant inefficiency)")
    print(f"  LR Statistic: {lr_pl_kumb['LR_stat']:.4f}")
    print(f"  P-value:      {lr_pl_kumb['p_value']:.4f}")
    print(f"  Reject H0:    {lr_pl_kumb['reject_H0']}")
    if lr_pl_kumb["reject_H0"]:
        print("  -> Flexible time pattern is significant")
    else:
        print("  -> Time-invariant model is adequate")

---

## 5. Returns to Scale Testing (20 min)

<a id="5-returns-to-scale-testing"></a>

### Returns to Scale

For a Cobb-Douglas production function:
$$\text{RTS} = \sum_j \beta_j$$

- **RTS = 1**: Constant returns to scale (CRS) -- doubling inputs doubles output
- **RTS > 1**: Increasing returns to scale (IRS) -- firms are too small
- **RTS < 1**: Decreasing returns to scale (DRS) -- firms are too large

### Wald Test

$$H_0: \text{RTS} = 1 \quad \text{vs} \quad H_1: \text{RTS} \neq 1$$

$$W = \frac{(\text{RTS} - 1)^2}{\text{Var}(\text{RTS})} \sim \chi^2(1)$$

where $\text{Var}(\text{RTS}) = \mathbf{1}' \text{Var}(\boldsymbol{\beta}) \mathbf{1}$ (delta method).

In [ ]:
# Returns to scale test using the best cross-section model
best_dist = best_aic if best_aic in models else next(iter(models.keys()))
result_best = models[best_dist]

rts_test = result_best.returns_to_scale_test(input_vars=exog_vars)

print("=" * 60)
print("RETURNS TO SCALE TEST")
print("=" * 60)
print(f"Model: {best_dist}")

# Input elasticities
elasticities = result_best.elasticities(input_vars=exog_vars)
print("\nInput Elasticities (Cobb-Douglas):")
for var, elas in elasticities.items():
    print(f"  {var:<15}: {elas:.4f}")

print("\nReturns to Scale:")
print(f"  RTS estimate: {rts_test['rts']:.4f}")
print(f"  Std Error:    {rts_test['rts_se']:.4f}")
print(f"  95% CI:       [{rts_test['ci'][0]:.4f}, {rts_test['ci'][1]:.4f}]")

print("\nWald Test (H0: RTS = 1):")
print(f"  Test Statistic: {rts_test['test_statistic']:.4f}")
print(f"  P-value:        {rts_test['pvalue']:.4f}")
print(f"  Conclusion:     {rts_test['conclusion']}")
print(f"\n{rts_test['interpretation']}")

In [ ]:
# Visualize RTS estimate with confidence interval
fig, ax = plt.subplots(figsize=(8, 5))

rts_val = rts_test["rts"]
rts_ci = rts_test["ci"]

# RTS point estimate and CI
ax.errorbar(
    rts_val,
    0,
    xerr=[[rts_val - rts_ci[0]], [rts_ci[1] - rts_val]],
    fmt="o",
    markersize=12,
    capsize=10,
    capthick=2,
    linewidth=2,
    color="steelblue",
    label=f"RTS = {rts_val:.3f}",
)

# Reference line at RTS = 1 (CRS)
ax.axvline(1, color="red", linewidth=2, linestyle="--", label="CRS (RTS = 1)")

# Shading
ax.axvspan(0, 1, alpha=0.1, color="orange", label="DRS region")
ax.axvspan(1, 2, alpha=0.1, color="green", label="IRS region")

ax.set_xlabel("Returns to Scale", fontsize=13)
ax.set_title(
    f"Returns to Scale Test\n(p-value = {rts_test['pvalue']:.4f})", fontsize=14, fontweight="bold"
)
ax.set_yticks([])
ax.legend(fontsize=10, loc="upper right")
ax.set_xlim(max(0, rts_ci[0] - 0.3), rts_ci[1] + 0.3)
ax.grid(True, alpha=0.3, axis="x")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "rts_confidence_interval.png", dpi=300, bbox_inches="tight")
plt.show()

print("Figure saved: rts_confidence_interval.png")

*Figure: Returns to scale point estimate with 95% confidence interval. The red dashed line at RTS=1 marks constant returns to scale. If the CI includes 1, we cannot reject CRS.*

---

## 6. Functional Form Testing (25 min)

<a id="6-functional-form-testing"></a>

### Cobb-Douglas vs Translog

**Cobb-Douglas** (constant elasticities):
$$\ln y = \beta_0 + \sum_j \beta_j \ln x_j + v - u$$

**Translog** (variable elasticities):
$$\ln y = \beta_0 + \sum_j \beta_j \ln x_j + \frac{1}{2} \sum_i \sum_j \beta_{ij} \ln x_i \ln x_j + v - u$$

### LR Test
$$H_0: \beta_{ij} = 0 \; \forall \; i,j \quad \text{(Cobb-Douglas)}$$
$$H_1: \text{At least one } \beta_{ij} \neq 0 \quad \text{(Translog)}$$

In [ ]:
# Step 1: Cobb-Douglas model (already estimated)
cd_result = models[best_dist]
print(f"Cobb-Douglas ({best_dist}):")
print(f"  Log-L: {cd_result.loglik:.4f}, AIC: {cd_result.aic:.4f}, BIC: {cd_result.bic:.4f}")

# Step 2: Create translog data
data_tl = add_translog(data, exog_vars)

# Identify the translog terms that were created
tl_new_cols = [c for c in data_tl.columns if c not in data.columns]
print(f"\nTranslog terms added: {tl_new_cols}")
print(f"Total regressors (CD): {len(exog_vars)}")
print(f"Total regressors (TL): {len(exog_vars) + len(tl_new_cols)}")

In [ ]:
# Step 3: Estimate translog model
tl_exog = exog_vars + tl_new_cols

try:
    tl_model = StochasticFrontier(
        data=data_tl,
        depvar="log_milk",
        exog=tl_exog,
        frontier="production",
        dist=best_dist,
    )
    tl_result = tl_model.fit()

    print(f"Translog ({best_dist}):")
    print(f"  Log-L: {tl_result.loglik:.4f}, AIC: {tl_result.aic:.4f}, BIC: {tl_result.bic:.4f}")
    print(f"  Converged: {tl_result.converged}")
    print(f"  N params (CD): {cd_result.nparams}, N params (TL): {tl_result.nparams}")
except Exception as e:
    print(f"Translog estimation failed: {e}")
    tl_result = None

In [ ]:
# Step 4: LR test for functional form
if tl_result is not None:
    ff_test = cd_result.compare_functional_form(tl_result)

    print("=" * 60)
    print("FUNCTIONAL FORM TEST: Cobb-Douglas vs Translog")
    print("=" * 60)
    print("H0: All interaction/squared terms = 0 (Cobb-Douglas)")
    print("H1: At least one interaction term != 0 (Translog)")
    print(f"\nLog-Likelihood (CD):  {ff_test['aic_cd'] / -2 + cd_result.nparams:.4f}")
    print(f"Log-Likelihood (TL):  {ff_test['aic_tl'] / -2 + tl_result.nparams:.4f}")
    print(f"\nLR Statistic: {ff_test['lr_statistic']:.4f}")
    print(f"Degrees of Freedom: {ff_test['df']}")
    print(f"P-value: {ff_test['pvalue']:.4f}")
    print(f"\nConclusion: {ff_test['conclusion']}")
    print(f"{ff_test['interpretation']}")
    print("\nInformation Criteria:")
    print(
        f"  AIC: CD = {ff_test['aic_cd']:.2f}, Translog = {ff_test['aic_tl']:.2f} (delta = {ff_test['delta_aic']:.2f})"
    )
    print(
        f"  BIC: CD = {ff_test['bic_cd']:.2f}, Translog = {ff_test['bic_tl']:.2f} (delta = {ff_test['delta_bic']:.2f})"
    )
else:
    print("Translog model not available. Skipping functional form test.")

---

## 7. Bootstrap Inference (30 min)

<a id="7-bootstrap-inference"></a>

### Why Bootstrap for SFA?

Standard asymptotic inference may be unreliable in SFA because:

1. **Small sample sizes**: Asymptotic theory requires large N
2. **Non-normality**: Efficiency estimators (JLMS, BC) are nonlinear transformations
3. **Boundary parameters**: $\sigma_u^2 \geq 0$ creates non-standard distributions
4. **Efficiency rankings**: Uncertainty in who is most/least efficient

### Bootstrap Methods

| Method | Description | When to Use |
|--------|-------------|------------|
| **Parametric** | Resample from fitted DGP | Correct error distribution assumed |
| **Pairs** | Resample $(y_i, \mathbf{x}_i)$ pairs | Distribution-free, robust |

In [ ]:
# Bootstrap parameters (use smaller n_boot for speed in tutorial)
boot = SFABootstrap(
    result=cd_result,
    method="parametric",
    n_boot=200,
    ci_level=0.95,
    seed=42,
    n_jobs=1,
)

boot_params = boot.bootstrap_parameters()

print("=" * 70)
print("BOOTSTRAP PARAMETER CONFIDENCE INTERVALS (Parametric, B=200)")
print("=" * 70)
display(boot_params["results_df"].round(4))
print(
    f"\nValid replications: {boot_params['n_valid']}/{boot_params['n_valid'] + boot_params['n_failed']}"
)

In [ ]:
# Bootstrap efficiency scores
boot_eff = boot.bootstrap_efficiency(estimator="bc")

print("=" * 70)
print("BOOTSTRAP EFFICIENCY CONFIDENCE INTERVALS (first 15 farms)")
print("=" * 70)
display(boot_eff.head(15).round(4))

In [ ]:
# Compare asymptotic vs bootstrap CIs
asym_eff = cd_result.efficiency(estimator="bc", ci_level=0.95)

n_compare = 20
comparison_ci = pd.DataFrame(
    {
        "Farm": range(1, n_compare + 1),
        "TE": asym_eff["efficiency"].values[:n_compare],
        "Asym_Lower": asym_eff["ci_lower"].values[:n_compare],
        "Asym_Upper": asym_eff["ci_upper"].values[:n_compare],
        "Boot_Lower": boot_eff["ci_lower"].values[:n_compare],
        "Boot_Upper": boot_eff["ci_upper"].values[:n_compare],
    }
)
comparison_ci["Asym_Width"] = comparison_ci["Asym_Upper"] - comparison_ci["Asym_Lower"]
comparison_ci["Boot_Width"] = comparison_ci["Boot_Upper"] - comparison_ci["Boot_Lower"]

print("=" * 80)
print("COMPARISON: Asymptotic vs Bootstrap CIs (first 20 farms)")
print("=" * 80)
display(comparison_ci.round(4))

print(f"\nMean CI width -- Asymptotic: {comparison_ci['Asym_Width'].mean():.4f}")
print(f"Mean CI width -- Bootstrap:  {comparison_ci['Boot_Width'].mean():.4f}")

In [ ]:
# Visualize asymptotic vs bootstrap CIs
fig, ax = plt.subplots(figsize=(12, 6))

n_plot = 20
entities = np.arange(1, n_plot + 1)
te = asym_eff["efficiency"].values[:n_plot]

asym_lower = asym_eff["ci_lower"].values[:n_plot]
asym_upper = asym_eff["ci_upper"].values[:n_plot]
boot_lower = boot_eff["ci_lower"].values[:n_plot]
boot_upper = boot_eff["ci_upper"].values[:n_plot]

# Asymptotic CIs
ax.errorbar(
    entities - 0.15,
    te,
    yerr=[te - asym_lower, asym_upper - te],
    fmt="o",
    capsize=4,
    capthick=1.5,
    markersize=6,
    color="steelblue",
    label="Asymptotic CI",
    alpha=0.8,
)

# Bootstrap CIs
ax.errorbar(
    entities + 0.15,
    te,
    yerr=[te - boot_lower, boot_upper - te],
    fmt="s",
    capsize=4,
    capthick=1.5,
    markersize=6,
    color="#e74c3c",
    label="Bootstrap CI",
    alpha=0.8,
)

ax.set_xlabel("Farm", fontsize=12)
ax.set_ylabel("Technical Efficiency", fontsize=12)
ax.set_title("Asymptotic vs Bootstrap Confidence Intervals", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xticks(entities)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "bootstrap_ci_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

print("Figure saved: bootstrap_ci_comparison.png")

*Figure: Comparison of asymptotic (circles) and bootstrap (squares) 95% confidence intervals for technical efficiency of the first 20 farms. Bootstrap CIs may differ in width and symmetry, providing a robustness check on the asymptotic intervals.*

---

## 8. Model Selection Workflow (25 min)

<a id="8-model-selection-workflow"></a>

### Systematic Model Selection

A principled model selection workflow follows these steps:

1. **Is inefficiency present?** (SFA vs OLS)
2. **Which distribution?** (half-normal, exponential, truncated normal)
3. **Functional form?** (Cobb-Douglas vs Translog)
4. **Returns to scale?** (CRS, IRS, DRS)

Each step uses the appropriate test (LR for nested, AIC/BIC for non-nested).

In [ ]:
def model_selection_workflow(data, depvar, exog, frontier_type="production"):
    """Complete step-by-step model selection workflow for SFA."""
    results = {}

    print("\n" + "=" * 80)
    print("STEP-BY-STEP MODEL SELECTION WORKFLOW")
    print("=" * 80)

    # -------------------------------------------------------
    # Step 1: Test for inefficiency
    # -------------------------------------------------------
    print("\nSTEP 1: Test for presence of inefficiency")
    print("-" * 60)

    X_step1 = sm.add_constant(data[exog])
    ols_step1 = sm.OLS(data[depvar], X_step1).fit()

    sfa_step1 = StochasticFrontier(
        data=data,
        depvar=depvar,
        exog=exog,
        frontier=frontier_type,
        dist="half_normal",
    ).fit()

    ineff_test = inefficiency_presence_test(
        loglik_sfa=sfa_step1.loglik,
        loglik_ols=ols_step1.llf,
        residuals_ols=ols_step1.resid.values,
        frontier_type=frontier_type,
    )

    print(f"  LR stat = {ineff_test['lr_statistic']:.4f}, p = {ineff_test['pvalue']:.4f}")
    if ineff_test["pvalue"] < 0.05:
        print("  -> Inefficiency is significant. Proceed with SFA.")
        results["step1"] = "SFA"
    else:
        print("  -> No evidence of inefficiency. Use OLS.")
        results["step1"] = "OLS"
        results["final_model"] = None
        return results

    # -------------------------------------------------------
    # Step 2: Select distribution
    # -------------------------------------------------------
    print("\nSTEP 2: Select distribution for inefficiency")
    print("-" * 60)

    dists = ["half_normal", "exponential", "truncated_normal"]
    dist_models = {}

    for dist in dists:
        try:
            dist_models[dist] = StochasticFrontier(
                data=data,
                depvar=depvar,
                exog=exog,
                frontier=frontier_type,
                dist=dist,
            ).fit()
        except Exception as e:
            print(f"  {dist}: Failed ({e})")

    aic_values = {d: dist_models[d].aic for d in dist_models}
    bic_values = {d: dist_models[d].bic for d in dist_models}
    best_dist_name = min(aic_values, key=aic_values.get)

    for d in dist_models:
        marker = " <-- BEST" if d == best_dist_name else ""
        print(f"  {d:<20}: AIC = {aic_values[d]:.2f}, BIC = {bic_values[d]:.2f}{marker}")

    results["step2"] = best_dist_name
    results["best_model"] = dist_models[best_dist_name]

    # -------------------------------------------------------
    # Step 3: Test functional form
    # -------------------------------------------------------
    print("\nSTEP 3: Test functional form (CD vs Translog)")
    print("-" * 60)

    cd_mod = dist_models[best_dist_name]
    try:
        data_tl_wf = add_translog(data, exog)
        tl_exog_wf = exog + [c for c in data_tl_wf.columns if c not in data.columns]

        tl_mod = StochasticFrontier(
            data=data_tl_wf,
            depvar=depvar,
            exog=tl_exog_wf,
            frontier=frontier_type,
            dist=best_dist_name,
        ).fit()

        ff_result = cd_mod.compare_functional_form(tl_mod)

        if ff_result["pvalue"] < 0.05:
            print(f"  LR = {ff_result['lr_statistic']:.4f}, p = {ff_result['pvalue']:.4f}")
            print("  -> Translog preferred")
            results["step3"] = "translog"
            results["best_model"] = tl_mod
        else:
            print(f"  LR = {ff_result['lr_statistic']:.4f}, p = {ff_result['pvalue']:.4f}")
            print("  -> Cobb-Douglas adequate")
            results["step3"] = "cobb_douglas"
    except Exception as e:
        print(f"  Translog not estimable: {e}")
        results["step3"] = "cobb_douglas"

    # -------------------------------------------------------
    # Step 4: Test returns to scale
    # -------------------------------------------------------
    print("\nSTEP 4: Test returns to scale")
    print("-" * 60)

    try:
        rts_result = results["best_model"].returns_to_scale_test(input_vars=exog)
        print(f"  RTS = {rts_result['rts']:.4f}, p = {rts_result['pvalue']:.4f}")
        print(f"  -> {rts_result['conclusion']}")
        results["step4"] = rts_result["conclusion"]
    except Exception as e:
        print(f"  RTS test failed: {e}")
        results["step4"] = "unknown"

    # -------------------------------------------------------
    # Summary
    # -------------------------------------------------------
    print("\n" + "=" * 80)
    print("FINAL MODEL SELECTION")
    print("=" * 80)
    print(f"  Inefficiency: {results['step1']}")
    print(f"  Distribution: {results['step2']}")
    print(f"  Functional form: {results['step3']}")
    print(f"  Returns to scale: {results['step4']}")
    print(f"  Mean TE: {results['best_model'].mean_efficiency:.4f}")

    results["final_model"] = results["best_model"]
    return results


# Run workflow
workflow_results = model_selection_workflow(
    data=data,
    depvar="log_milk",
    exog=exog_vars,
    frontier_type="production",
)

In [ ]:
# Create a model selection flowchart using matplotlib
fig, ax = plt.subplots(figsize=(12, 8))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis("off")


# Helper to draw a box
def draw_box(ax, x, y, text, color="lightblue", width=3, height=0.8):
    rect = plt.Rectangle(
        (x - width / 2, y - height / 2),
        width,
        height,
        facecolor=color,
        edgecolor="black",
        linewidth=1.5,
        zorder=2,
    )
    ax.add_patch(rect)
    ax.text(x, y, text, ha="center", va="center", fontsize=9, fontweight="bold", zorder=3)


def draw_arrow(ax, x1, y1, x2, y2, text="", color="black"):
    ax.annotate(
        "", xy=(x2, y2), xytext=(x1, y1), arrowprops={"arrowstyle": "->", "color": color, "lw": 1.5}
    )
    if text:
        mx, my = (x1 + x2) / 2, (y1 + y2) / 2
        ax.text(mx + 0.15, my, text, fontsize=8, color=color)


# Step 1
draw_box(ax, 5, 9, "Step 1: Inefficiency Test\n(LR with mixed chi-sq)", "lightyellow")
draw_arrow(ax, 5, 8.6, 5, 7.9)

# Step 2
step2_color = "#c8e6c9" if workflow_results.get("step1") == "SFA" else "#ffcdd2"
draw_box(ax, 5, 7.5, f"Step 2: Distribution\n({workflow_results.get('step2', '?')})", step2_color)
draw_arrow(ax, 5, 7.1, 5, 6.4)

# Step 3
draw_box(ax, 5, 6.0, f"Step 3: Functional Form\n({workflow_results.get('step3', '?')})", "#bbdefb")
draw_arrow(ax, 5, 5.6, 5, 4.9)

# Step 4
draw_box(ax, 5, 4.5, f"Step 4: Returns to Scale\n({workflow_results.get('step4', '?')})", "#e1bee7")
draw_arrow(ax, 5, 4.1, 5, 3.4)

# Final model
final_te = workflow_results.get("final_model")
te_str = f"Mean TE = {final_te.mean_efficiency:.3f}" if final_te else "N/A"
draw_box(ax, 5, 3.0, f"Final Model\n{te_str}", "#fff9c4", width=3.5)

ax.set_title("Model Selection Flowchart", fontsize=16, fontweight="bold", pad=20)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "model_selection_flowchart.png", dpi=300, bbox_inches="tight")
plt.show()

print("Figure saved: model_selection_flowchart.png")

*Figure: Flowchart summarizing the model selection decisions. Each box shows the test performed and the outcome, leading to the final preferred model specification.*

---

## 9. Export Results

<a id="9-export-results"></a>

In [ ]:
# Export distribution comparison table
dist_comparison_export = dist_comparison[
    ["Distribution", "Log-Likelihood", "AIC", "BIC", "σ_v", "σ_u", "Mean Efficiency"]
]

dist_latex = dist_comparison_export.to_latex(
    index=False,
    float_format="%.4f",
    caption="Distribution Comparison for Dairy Farm Data",
    label="tab:dist_comparison",
)
with open(TABLES_DIR_LATEX / "05_distribution_comparison.tex", "w") as f:
    f.write(dist_latex)

dist_html = dist_comparison_export.to_html(index=False, float_format="%.4f")
with open(TABLES_DIR_HTML / "05_distribution_comparison.html", "w") as f:
    f.write(dist_html)

print("Distribution comparison tables saved.")

# Export panel model comparison
panel_latex = panel_comparison.to_latex(
    index=False,
    float_format="%.4f",
    caption="Panel Model Comparison for Telecom Data",
    label="tab:panel_comparison",
)
with open(TABLES_DIR_LATEX / "05_panel_model_comparison.tex", "w") as f:
    f.write(panel_latex)

panel_html = panel_comparison.to_html(index=False, float_format="%.4f")
with open(TABLES_DIR_HTML / "05_panel_model_comparison.html", "w") as f:
    f.write(panel_html)

print("Panel model comparison tables saved.")

# Export LR tests summary
lr_tests_data = []
lr_tests_data.append(
    {
        "Test": "Inefficiency (SFA vs OLS)",
        "LR Statistic": test_result["lr_statistic"],
        "DF": test_result["df"],
        "P-value": test_result["pvalue"],
        "Conclusion": test_result["conclusion"],
    }
)
if "half_normal" in models and "truncated_normal" in models:
    lr_tests_data.append(
        {
            "Test": "HN vs TN",
            "LR Statistic": lr_hn_tn["lr_statistic"],
            "DF": lr_hn_tn.get("df", 1),
            "P-value": lr_hn_tn["pvalue"],
            "Conclusion": lr_hn_tn["conclusion"],
        }
    )
if tl_result is not None:
    lr_tests_data.append(
        {
            "Test": "CD vs Translog",
            "LR Statistic": ff_test["lr_statistic"],
            "DF": ff_test["df"],
            "P-value": ff_test["pvalue"],
            "Conclusion": ff_test["conclusion"],
        }
    )

lr_tests_df = pd.DataFrame(lr_tests_data)

lr_latex = lr_tests_df.to_latex(
    index=False,
    float_format="%.4f",
    caption="Summary of Likelihood Ratio Tests",
    label="tab:lr_tests",
)
with open(TABLES_DIR_LATEX / "05_lr_tests_summary.tex", "w") as f:
    f.write(lr_latex)

lr_html = lr_tests_df.to_html(index=False, float_format="%.4f")
with open(TABLES_DIR_HTML / "05_lr_tests_summary.html", "w") as f:
    f.write(lr_html)

print("LR tests summary tables saved.")

# Export bootstrap results
boot_latex = boot_params["results_df"].to_latex(
    index=False,
    float_format="%.4f",
    caption="Bootstrap Parameter Estimates",
    label="tab:bootstrap_params",
)
with open(TABLES_DIR_LATEX / "05_bootstrap_results.tex", "w") as f:
    f.write(boot_latex)

boot_html = boot_params["results_df"].to_html(index=False, float_format="%.4f")
with open(TABLES_DIR_HTML / "05_bootstrap_results.html", "w") as f:
    f.write(boot_html)

print("Bootstrap results tables saved.")

# Export model selection workflow summary
workflow_df = pd.DataFrame(
    {
        "Step": ["1. Inefficiency", "2. Distribution", "3. Functional Form", "4. Returns to Scale"],
        "Test": ["Mixed chi-sq LR", "AIC/BIC", "LR test", "Wald test"],
        "Decision": [
            workflow_results.get("step1", ""),
            workflow_results.get("step2", ""),
            workflow_results.get("step3", ""),
            workflow_results.get("step4", ""),
        ],
    }
)

wf_latex = workflow_df.to_latex(
    index=False,
    caption="Model Selection Workflow Results",
    label="tab:workflow",
)
with open(TABLES_DIR_LATEX / "05_model_selection_workflow.tex", "w") as f:
    f.write(wf_latex)

wf_html = workflow_df.to_html(index=False)
with open(TABLES_DIR_HTML / "05_model_selection_workflow.html", "w") as f:
    f.write(wf_html)

print("Model selection workflow tables saved.")
print("\nAll tables exported successfully!")

---

## 10. Exercises

<a id="10-exercises"></a>

### Exercise 1: Complete Model Selection (Easy)

Using the dairy farm data, follow the model selection workflow step-by-step and document your decisions.

**Tasks**:
1. Test for inefficiency presence using the **exponential** distribution
2. Compare half-normal, exponential, and truncated normal using AIC/BIC
3. For the best distribution, test Cobb-Douglas vs Translog
4. Test returns to scale for the final model
5. Report your final model specification and justify each choice

**Hints**:
- Use `inefficiency_presence_test()` with `distribution='exponential'`
- Use `compare_distributions()` or estimate each model separately
- Use `compare_functional_form()` for the CD vs Translog test
- Use `returns_to_scale_test()` for the RTS test

In [ ]:
# Exercise 1: Your code here

# Step 1: Test for inefficiency with exponential
# sfa_exp = StochasticFrontier(
#     data=data, depvar='log_milk', exog=exog_vars,
#     frontier='production', dist='exponential',
# ).fit()
# ineff = inefficiency_presence_test(...)

# Step 2: Compare distributions
# dist_comp = sfa_exp.compare_distributions(distributions=[...])

# Step 3: Functional form test
# data_tl = add_translog(data, exog_vars)
# ...

# Step 4: RTS test
# rts = result.returns_to_scale_test(input_vars=exog_vars)

# Step 5: Report final specification

### Exercise 2: Bootstrap Robustness (Medium)

Compare the robustness of efficiency estimates across bootstrap methods.

**Tasks**:
1. Run **parametric** bootstrap with B=200 on the best model
2. Run **pairs** bootstrap with B=200 on the same model
3. Compare the bootstrap confidence intervals for the top 10 and bottom 10 farms
4. Compute Spearman rank correlation of efficiency rankings between methods
5. Discuss: Are the most/least efficient farms consistent across methods?

**Hints**:
- Use `SFABootstrap(result, method='parametric', ...)` and `method='pairs'`
- Compare `bootstrap_efficiency()` results
- Use `scipy.stats.spearmanr()` for rank correlation

In [ ]:
# Exercise 2: Your code here

# Step 1: Parametric bootstrap
# boot_param = SFABootstrap(result, method='parametric', n_boot=200, seed=42, n_jobs=1)
# eff_param = boot_param.bootstrap_efficiency(estimator='bc')

# Step 2: Pairs bootstrap
# boot_pairs = SFABootstrap(result, method='pairs', n_boot=200, seed=42, n_jobs=1)
# eff_pairs = boot_pairs.bootstrap_efficiency(estimator='bc')

# Step 3: Compare top 10 and bottom 10

# Step 4: Rank correlation

# Step 5: Discussion

### Exercise 3: Specification Testing (Hard)

Conduct a comprehensive specification analysis for the telecom panel data.

**Tasks**:
1. Estimate Pitt-Lee, BC92, and Kumbhakar 1990 with both half-normal and exponential distributions (6 models total)
2. Create a comprehensive comparison table with Log-L, AIC, BIC, Mean TE
3. Perform all pairwise LR tests for nested models
4. Test for returns to scale in the best panel model
5. Write a brief model selection report summarizing all findings

**Hints**:
- Use the `model_type` parameter to specify panel models
- `lr_test_bc92_eta_constant()` and `lr_test_kumbhakar_constant()` for nested panel tests
- For non-nested comparisons (e.g., half-normal vs exponential), use AIC/BIC

In [ ]:
# Exercise 3: Your code here

# Step 1: Estimate 6 models (3 panel types x 2 distributions)
# panel_types = ['pitt_lee', 'bc92', 'kumbhakar_1990']
# dists = ['half_normal', 'exponential']
# all_panel_models = {}
# for ptype in panel_types:
#     for dist in dists:
#         name = f'{ptype}_{dist}'
#         model = StochasticFrontier(
#             data=panel_data, depvar='log_output', exog=panel_exog,
#             entity='firm_id', time='year',
#             frontier='production', dist=dist, model_type=ptype,
#         )
#         all_panel_models[name] = model.fit()

# Step 2: Comparison table

# Step 3: LR tests

# Step 4: RTS test

# Step 5: Report

---

## 11. Summary

<a id="11-summary"></a>

### What You Have Learned

1. **Inefficiency presence test** using the mixed chi-square LR test (Kodde-Palm critical values)
2. **Distribution selection** via AIC/BIC for non-nested models and LR test for nested models
3. **Panel model selection** comparing Pitt-Lee, BC92, and Kumbhakar specifications
4. **Returns to scale testing** using the Wald test for H0: RTS = 1
5. **Functional form testing** comparing Cobb-Douglas and Translog via LR test
6. **Bootstrap inference** for robust confidence intervals on parameters and efficiency
7. **Systematic model selection workflow** combining all tests into a principled decision process

### Key Formulas

| Test | Formula | Distribution |
|------|---------|-------------|
| Inefficiency LR | $LR = 2(\ln L_{SFA} - \ln L_{OLS})$ | Mixed $\chi^2$ |
| Nested LR | $LR = 2(\ln L_U - \ln L_R)$ | $\chi^2(q)$ |
| AIC | $-2\ln L + 2k$ | N/A |
| BIC | $-2\ln L + k\ln(N)$ | N/A |
| Wald (RTS) | $(\text{RTS}-1)^2 / \text{Var}(\text{RTS})$ | $\chi^2(1)$ |

### Common Pitfalls

1. Using standard $\chi^2$ instead of mixed $\chi^2$ for the inefficiency test
2. Comparing non-nested models with LR tests (use AIC/BIC instead)
3. Ignoring BIC's stronger parsimony penalty in small samples
4. Not checking skewness before fitting SFA
5. Using too few bootstrap replications (use B >= 500 for publication)

### Next Steps

In Tutorial 06 (Complete Case Study), we will:
- Apply all techniques from tutorials 01-05 to a real-world dataset
- Conduct end-to-end frontier analysis with full reporting

---

## 12. References

<a id="12-references"></a>

1. **Coelli, T.J. (1995)**. "Estimators and hypothesis tests for a stochastic frontier function: A Monte Carlo analysis." *Journal of Productivity Analysis*, 6(4), 247-268.

2. **Kodde, D.A., & Palm, F.C. (1986)**. "Wald criteria for jointly testing equality and inequality restrictions." *Econometrica*, 54(5), 1243-1248.

3. **Battese, G.E., & Coelli, T.J. (1992)**. "Frontier production functions, technical efficiency and panel data: With application to paddy farmers in India." *Journal of Productivity Analysis*, 3(1), 153-169.

4. **Kumbhakar, S.C. (1990)**. "Production frontiers, panel data, and time-varying technical inefficiency." *Journal of Econometrics*, 46(1-2), 201-211.

5. **Kumbhakar, S.C., & Lovell, C.A.K. (2000)**. *Stochastic Frontier Analysis*. Cambridge University Press.

---

*This tutorial is part of the PanelBox Stochastic Frontier Analysis module. For questions or suggestions, please open an issue on the project repository.*